In [1]:
import boto3
import json
import datetime
from IPython.display import display, JSON

MODEL_ID = "us.anthropic.claude-sonnet-4-6"
bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

bedrock_agent = boto3.client("bedrock-agent", region_name="us-east-1")

In [2]:
temperature = 0.7

inference_config = {"temperature": temperature}

system_prompts = [
    {
        "text": "You are a virtual travel assistant that suggests destinations based on user preferences."
        + "Only return destination names and a brief description."
    }
]

messages = []

message_1 = {
    "role": "user",
    "content": [{"text": "Create a list of 3 travel destinations."}],
}

messages.append(message_1)

bedrock = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config,
)


def print_response(response):
    model_response = (
        response.get("output", {})
        .get("message", {})
        .get("content", [{}])[0]
        .get("text", "")
    )

    print("✈️ Your suggested travel destinations:")
    print(model_response)


print_response(response)

✈️ Your suggested travel destinations:
Here are 3 travel destinations you might enjoy:

1. **Kyoto, Japan** - A serene city filled with ancient temples, traditional tea houses, and stunning cherry blossom gardens, perfect for culture lovers.

2. **Santorini, Greece** - A breathtaking island featuring iconic white-washed buildings, crystal-clear blue waters, and spectacular sunsets over the Aegean Sea.

3. **Patagonia, Argentina** - A dramatic wilderness destination offering towering glaciers, rugged mountain peaks, and vast open landscapes ideal for adventure seekers.


In [3]:
message_2 = {
    "role": "user",
    "content": [
        {
            "text": "Only suggest travel locations that are no more than one short flight away."
        }
    ],
}

messages.append(message_2)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config,
)

print_response(response)

✈️ Your suggested travel destinations:
Here are 3 great travel destinations within a short flight away:

1. **Barcelona, Spain** - A vibrant city known for its stunning architecture, beautiful beaches, and world-class cuisine.

2. **Amsterdam, Netherlands** - A charming city famous for its scenic canals, historic museums, and colorful tulip fields.

3. **Prague, Czech Republic** - A fairytale city boasting breathtaking medieval architecture, a rich cultural history, and a lively atmosphere.

*Note: These suggestions are based on a general European starting point. Destinations may vary depending on your specific location.*


In [4]:
try:
    response = bedrock_agent.create_prompt(
        name="Travel-Agent-Prompt",
        description="Checks if all trip information has been provided.",
        variants=[
            {
                "name": "Variant1",
                "modelId": MODEL_ID,
                "templateType": "CHAT",
                "inferenceConfiguration": {"text": {"temperature": 0.4}},
                "templateConfiguration": {
                    "chat": {
                        "system": [
                            {
                                "text": """You are a travel agent evaluating trip requests for custom itineraries. 
                                Review the message carefully and answer YES or NO to the following screening questions. 
                                Be strict—if any detail is missing or unclear, answer NO.

                                A) Is the destination clearly stated?
                                B) Are the travel dates within a reasonable range (not last−minute or over a year away)?
                                C) Does the request avoid high−risk or restricted activities (e.g., extreme sports, off−grid travel)?
                                D) Is there any mention of a valid passport or travel documentation?
                                E) Is there enough information to follow up with a proposed itinerary?"""
                            }
                        ],
                        "messages": [
                            {
                                "role": "user",
                                "content": [
                                    {"text": "Trip request: {{event_request}}"}
                                ],
                            }
                        ],
                        "inputVariables": [{"name": "event_request"}],
                    }
                },
            }
        ],
    )
    print("Created!")
    prompt_arn = response.get("arn")
except bedrock.exceptions.ConflictException as e:
    print("Already exists!")
    response = bedrock.list_prompts()
    prompt = next(
        (
            prompt
            for prompt in response["promptSummaries"]
            if prompt["name"] == "TripBooker_xyz"
        ),
        None,
    )
    prompt_arn = prompt["arn"]

prompt_arn

Created!


'arn:aws:bedrock:us-east-1:623271127785:prompt/CISRH3Q0V3'

In [5]:
response = bedrock.converse(
    modelId=prompt_arn,
    promptVariables={"event_request": {"text": """
                Hi there! I'm planning a trip to Italy with my partner and would love some help organizing the itinerary. We're hoping to travel between September 10–20 this year, ideally flying into Rome and spending a few days in Florence and Venice as well. We’d love recommendations on tours, cultural sites, and good local restaurants. We’re not interested in anything risky like skydiving or hiking remote trails — just want a relaxing and enriching experience. We both have valid passports. Let me know what other details you need!
                """}},
)
print(response["output"]["message"]["content"][0]["text"])

Here is my evaluation of this trip request:

---

**A) Is the destination clearly stated?**
✅ **YES** — Italy is clearly stated, with specific cities: Rome, Florence, and Venice.

**B) Are the travel dates within a reasonable range?**
✅ **YES** — September 10–20 is a well-defined, 10-day window that is neither last-minute nor excessively far in advance.

**C) Does the request avoid high-risk or restricted activities?**
✅ **YES** — The travelers explicitly state they want a relaxing, cultural experience and specifically rule out risky activities.

**D) Is there any mention of a valid passport or travel documentation?**
✅ **YES** — Both travelers are confirmed to have valid passports.

**E) Is there enough information to follow up with a proposed itinerary?**
✅ **YES** — Destinations, dates, interests (tours, culture, dining), travel style, and party size (2 people) are all provided.

---

**Overall Assessment: ✅ APPROVED**

This is a well-structured, complete request. All five criteria 